<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex04-perceptron-to-mlp/Ex04_05_overfit_then_regularise_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference text — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_04 · Notebook 05 — Overfit, Then Regularise

**Deep Learning for Engineering · Aalborg University · Part 1**

This notebook uses **the eleven points from L4.2 slides 11 to 13** — the same
eleven numbers, copied out of the lecture's own generator. The figure you produce
in section 2 is the figure that was on the screen in the lecture theatre, and the
point of that continuity is that you should recognise it.

## What the lecture did with them

Eleven noisy measurements of a smooth process, fitted three ways:

- **too rigid** — a straight line. Two parameters, and the truth has a hump, so
  no setting of them gets close. Large training error, large held-out error, and
  both for the same reason.
- **about right** — a model with enough flexibility to follow the hump and not
  enough to follow the noise.
- **too flexible** — a curve that passes through every single sample and
  oscillates violently between them.

Capacity is a dial, not a target. Too little and you cannot represent the truth;
too much and you represent the noise. There is no formula for the middle, and the
engineering is entirely in finding it.

## What you will do

1. fit a deliberately enormous network to eleven points and drive the training
   loss to zero,
2. look at the held-out error and at the curve between the points,
3. add **weight decay** and sweep it,
4. add **early stopping**,
5. measure how much of the damage each one recovers.

**Every run in this notebook is reported with its training and validation curves
on the same axes.** No exceptions. That is a standing requirement of this course,
and this is the notebook that shows you why: the training curve of the
overfitted model is the most reassuring figure in the whole exercise, and it is
lying to you.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_4_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex04-perceptron-to-mlp/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
import torch

import Ex_4_core as core

x_train, y_train = core.lecture_dataset()          # the eleven points
x_val, y_val = core.lecture_validation(n=40)       # a second campaign
xs = np.linspace(0.0, 1.0, 600)

NOISE_FLOOR = core.LECTURE_NOISE_SD ** 2

print("training set:", len(x_train), "points")
print("held-out set:", len(x_val), "points")
print(f"noise variance (the best MSE anything can reach): {NOISE_FLOOR:.5f}")
print()
print("the eleven points, verbatim from the lecture:")
for xi, yi in zip(x_train, y_train):
    print(f"   x = {xi:.1f}   y = {yi:+.4f}")

**What you should see.** Eleven points at $x = 0.0, 0.1, \dots, 1.0$, starting at
$y = 0.3600$ and ending at $y = 0.4644$, and a noise floor of `0.00490`.

Those eleven numbers are not regenerated from a seed. They are the lecture's, and
`core.LECTURE_NOISE` holds the eleven noise values that were added to the truth
to make them.

---

## 1 · Fit a model that is much too large

Build a network with two hidden layers of 64 tanh units and train it far longer
than the problem deserves.

`tanh` rather than `relu` here, for two reasons. It is what Part 2 of the course
uses; and an overfitted tanh network oscillates smoothly between the data points,
which is what the lecture's slide 13 shows, where an overfitted ReLU network
would simply join the dots with straight lines and look deceptively reasonable.

### Your turn

Train the overfitting run.

- `core.set_seed(0)`
- `core.MLP(hidden=(64, 64), activation="tanh")`
- `core.train(..., x_val=x_val, y_val=y_val, epochs=6000, lr=1e-2,
  keep_best=True)`

`keep_best=True` records the weights at the best validation epoch. You will not
use them until section 4, but recording them costs nothing and it means the early
stopping experiment uses this exact run rather than a new one.

In [ ]:
# TODO 1 --- an over-large network on eleven points -----------------------------------------
# Two `...` to replace, one per line:
#   model_big    ->  core.MLP(hidden=(64, 64), activation="tanh")
#   history_big  ->  core.train(model_big, x_train, y_train, x_val=x_val, y_val=y_val,
#                               epochs=6000, lr=1e-2, keep_best=True)
core.set_seed(0)
model_big = ...                                   # <- core.MLP(hidden=(64, 64), activation="tanh")
print("parameters:", core.count_parameters(model_big))
history_big = ...                                 # <- core.train(model_big, x_train, y_train, x_val=x_val, y_val=y_val, epochs=6000, lr=1e-2, keep_best=True)
# ------------------------------------------------------------------------------

In [ ]:
train_big = core.evaluate(model_big, x_train, y_train)
val_big = core.evaluate(model_big, x_val, y_val)

print(f"parameters            : {core.count_parameters(model_big)}")
print(f"training points       : {len(x_train)}")
print(f"parameters per point  : {core.count_parameters(model_big) / len(x_train):.0f}")
print()
print(f"final training MSE    : {train_big:.8f}")
print(f"final validation MSE  : {val_big:.5f}")
print(f"noise floor           : {NOISE_FLOOR:.5f}")
print(f"best validation MSE   : {history_big['best_val']:.5f}"
      f"  at epoch {history_big['best_epoch']}")

**What you should see.** **4353** parameters for eleven data points — about four
hundred parameters per point. A final training MSE of order $10^{-7}$ or smaller,
which is zero for any practical purpose. A final validation MSE somewhere around
0.006 or 0.007. And a *best* validation MSE of about 0.0036, reached within the
first few hundred epochs and never recovered afterwards.

Sit with those numbers for a moment. The training error is roughly fifty thousand
times smaller than the variance of the noise in the measurements. That is not a
good model; it is a model that has memorised eleven numbers, including the part of
each one that was instrument error.

Note also that the best validation MSE is *below* the noise floor. That is not a
model beating the noise — it is a forty-point sample of a noisy quantity
fluctuating below its expected value. It is a reminder that a validation set of
forty points is itself a noisy instrument, and section 3 adds a second measure
that does not have this problem.

---

## 2 · The two figures

The first is the loss curves. The second is the fitted function.

In [ ]:
core.plot_curves(history_big,
                 title="Too flexible — 4353 parameters, eleven points")
plt.show()

**What you should see.** A blue training curve that falls smoothly across five
orders of magnitude and keeps going, and a red validation curve that falls for a
few hundred epochs, reaches a minimum, and then **rises steadily for the rest of
the run**. The green dashed line marks the minimum.

This figure is the reason for the rule about plotting both curves. Cover the red
one with your hand: what is left is a textbook picture of successful training,
and it would be reported as such. Everything that has gone wrong is in the curve
that costs nothing to add and that a training script does not produce unless you
ask it to.

Note also *when* the best validation epoch is — usually within the first few
hundred epochs of a six-thousand-epoch run. The remaining ninety-five per cent of
the training made the model worse, at increasing computational cost.

In [ ]:
core.plot_fit(x_train, y_train, {"overfitted": core.predict(model_big, xs)},
              x_curve=xs, truth=core.lecture_truth, x_val=x_val, y_val=y_val,
              title="L4.2 slide 13, on your own screen", ylim=(-0.3, 1.6))
plt.show()

y_curve = core.predict(model_big, xs)
worst_residual = np.abs(core.predict(model_big, x_train) - y_train).max()
worst_gap = np.abs(y_curve - core.lecture_truth(xs)).max()

print(f"largest residual on the eleven training points : {worst_residual:.2e}")
print(f"largest gap between the fitted curve and the truth: {worst_gap:.3f}")
print(f"the noise it was chasing had sd                  : {core.LECTURE_NOISE_SD}")

**What you should see.** A curve that passes through every one of the eleven black
markers — the largest training residual is of order $10^{-4}$, a hundred times
smaller than the instrument noise — and wanders away from the grey truth between
them. The largest gap between the fit and the truth is around **0.16**, which is
more than twice the standard deviation of the noise the model was chasing.

That is the shape of the failure: exact where it was measured, and wrong by more
than the measurement error everywhere in between.

That is slide 13. The lecture drew it with an interpolation scheme chosen to
misbehave; you have produced it with a perfectly ordinary network, trained
perfectly correctly, on the lecture's own data.

**Nothing here is a bug.** The optimiser did exactly what it was asked: minimise
the mean squared error on eleven points. It succeeded completely. The problem is
that "mean squared error on eleven points" was never what anybody wanted, and no
part of the training procedure knew that.

---

## 3 · Measuring the damage

Three numbers, because "it looks wrong" is not a measurement.

**Held-out error.** The mean squared error on the forty points the model never
saw. The honest estimate of what the model is worth, and the only one available
in a real project. It has a floor at the noise variance, 0.0049, and it is itself
noisy, being an average over forty samples.

**Error against the truth.** The mean squared difference between the fitted curve
and `core.lecture_truth` on a dense grid. This is what you actually want to know
and what you can never have: it exists here only because the data is synthetic.
Its floor is zero, which makes ratios between models meaningful — so it is the
one used to score the fixes in section 6.

**Total variation.** How far the curve travels vertically as $x$ goes from 0 to 1:

$$ \mathrm{TV} \;=\; \int_0^1 \left| f'(x) \right| \, \mathrm{d}x
   \;\approx\; \sum_i \bigl| f(x_{i+1}) - f(x_i) \bigr| $$

A model that follows the truth has roughly the truth's total variation. A model
that oscillates between the data points has a much larger one. It is a crude
measure of wiggliness and it is enough.

### Your turn

Implement total variation, and report both numbers for the overfitted model
alongside the same numbers for the truth.

In [ ]:
# TODO 2 --- total variation ---------------------------------------------------------------
# One `...` to replace, in the return line:  float(np.abs(np.diff(y)).sum())
def total_variation(y):
    """The sum of |differences| between neighbouring values of a sampled curve."""
    return ...                                    # <- float(np.abs(np.diff(y)).sum())
# ------------------------------------------------------------------------------

In [ ]:
def truth_error(model):
    # Mean squared error against the true curve, on the dense grid.
    # Available only because this dataset is synthetic.
    return core.mse(core.predict(model, xs), core.lecture_truth(xs))

tv_truth = total_variation(core.lecture_truth(xs))
tv_big = total_variation(core.predict(model_big, xs))
terr_big = truth_error(model_big)

print(f"total variation of the truth    : {tv_truth:.3f}")
print(f"total variation of the overfit  : {tv_big:.3f}"
      f"   ({tv_big / tv_truth:.1f} times the truth)")
print()
print(f"held-out MSE of the overfit     : {val_big:.5f}"
      f"   (noise floor {NOISE_FLOOR:.5f})")
print(f"error against the truth         : {terr_big:.5f}   (floor 0)")

**What you should see.** A truth with total variation about **0.93**, an
overfitted model with about **1.6** — the fitted curve travels roughly seventy per
cent further than the process it is modelling, which is the wiggle, measured. A
held-out MSE of about 0.0067 against a floor of 0.0049, and an error against the
truth of about **0.005**.

Note how undramatic the held-out number looks: 0.0067 against a floor of 0.0049
is less than fifty per cent above the floor, because most of that error is noise
in the held-out samples themselves and only a small part is the model's fault.
The error against the truth separates the two, and it is the number the rest of
the notebook tries to reduce.

---

## 4 · Fix one: weight decay

Weight decay adds a penalty on the size of the parameters to the loss:

$$ \tilde{\mathcal{L}}(\theta) \;=\; \mathcal{L}(\theta)
   \;+\; \lambda \sum_j \theta_j^2 $$

The mechanism is worth stating in words, because "it penalises complexity" is
too vague to be useful. Large weights are what let a network produce a large
change in output for a small change in input — a steep segment, a sharp
oscillation. Penalising the sum of their squares makes steepness *expensive*, so
the optimiser buys it only where the data pays for it. Wiggles that exist purely
to pass through a noisy point stop being worth their cost.

$\lambda$ is not a parameter you can fit, because the training loss always
prefers $\lambda = 0$. It has to be chosen on data the fit did not see.

In PyTorch, `torch.optim.Adam(..., weight_decay=λ)`, which `core.train` exposes
directly.

### Your turn

Sweep $\lambda$ over `[0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]`. For each value,
seed with 0, build a fresh `MLP(hidden=(64, 64), activation="tanh")`, and train
for 6000 epochs at `lr=1e-2` with that weight decay, keeping the history.

Fresh model each time: reusing the overfitted one starts from a memorised
solution and measures nothing.

In [ ]:
LAMBDAS = [0.0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]

# TODO 3 --- the weight-decay sweep ----------------------------------------------------------
# Two `...` to replace, inside the loop:
#   line 1  ->  core.MLP(hidden=(64, 64), activation="tanh")        a FRESH model each time
#   line 2  ->  core.train(model, x_train, y_train, x_val=x_val, y_val=y_val,
#                          epochs=6000, lr=1e-2, weight_decay=lam, keep_best=True)
wd_runs, wd_models = {}, {}
for lam in LAMBDAS:
    core.set_seed(0)
    model = ...                                   # <- core.MLP(hidden=(64, 64), activation="tanh")
    wd_runs[lam] = ...                            # <- core.train(model, x_train, y_train, x_val=x_val, y_val=y_val, epochs=6000, lr=1e-2, weight_decay=lam, keep_best=True)
    wd_models[lam] = model
    print(f"weight decay {lam:g}: best val {wd_runs[lam]['best_val']:.5f}")
# ------------------------------------------------------------------------------

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16.0, 7.0), sharex=True, sharey=True)
for ax, lam in zip(axes.ravel(), LAMBDAS):
    core.plot_curves(wd_runs[lam], ax=ax, title="")
    ax.get_legend().remove()
    ax.set_title(f"weight decay = {lam:g}", fontsize=10)
axes.ravel()[-1].axis("off")
axes[0, 0].legend(frameon=False, fontsize=8)
fig.suptitle("Weight decay: training (blue) and validation (red), every run",
             fontsize=12)
plt.tight_layout()
plt.show()

**What you should see.** Seven panels. At $\lambda = 0$ the two curves separate
dramatically. As $\lambda$ grows they close up: the blue curve stops reaching
$10^{-7}$ and the red curve stops rising. At the largest value the two lie almost
on top of each other and **both** are high — the penalty now dominates the data
and the model is too rigid to fit even the hump.

That progression, left to right, is the capacity dial from L4.2 slide 10, turned
by a single number.

In [ ]:
rows = []
for lam in LAMBDAS:
    m = wd_models[lam]
    tr = core.evaluate(m, x_train, y_train)
    va = core.evaluate(m, x_val, y_val)
    tv = total_variation(core.predict(m, xs))
    rows.append([f"{lam:g}", f"{tr:.6f}", f"{va:.5f}", f"{truth_error(m):.5f}",
                 f"{tv:.2f}"])
print(core.error_table(rows, ["weight decay", "training MSE", "validation MSE",
                              "error vs truth", "total variation"]))
print(f"\nnoise floor {NOISE_FLOOR:.5f},  truth total variation {tv_truth:.2f}")

best_lambda = min(LAMBDAS, key=lambda l: core.evaluate(wd_models[l], x_val, y_val))
print(f"\nbest weight decay on the held-out set: {best_lambda:g}")

**What you should see.** A training error that rises monotonically with
$\lambda$, a validation error that falls and then rises again — the classical U —
and a total variation that falls from about 1.6 towards the truth's 0.93 and then
past it, collapsing towards zero at the largest values.

Typical numbers: validation error around 0.0067 at $\lambda = 0$, a minimum near
0.0040 at $\lambda = 10^{-3}$, and back up above 0.02 at $\lambda = 10^{-1}$. The
error against the truth tells the same story far more sharply — about 0.005,
falling to roughly 0.0002 at the best $\lambda$, then rising to 0.028 — because it
does not have forty points of measurement noise sitting on top of it.

The best $\lambda$ is usually $10^{-4}$ or $10^{-3}$. Three features of the table
deserve comment in your report:

- **The best model on validation is not the best on training.** It is, in fact,
  several thousand times worse on training. If you had chosen by training error
  you would have chosen the worst model in the table.
- **The U is asymmetric.** Going one step too small costs you a little; going one
  or two steps too large costs you a lot, because the model can no longer
  represent the hump at all. When in doubt, err on the small side and rely on
  early stopping.
- **The two error columns do not agree perfectly on the winner.** They are
  measuring different things, and the validation column has noise in it that the
  truth column does not. In a real project you only have the noisy one, which is
  worth remembering before treating a five per cent difference as a result.

---

## 5 · Fix two: early stopping

The second fix costs nothing at all. You already have it: `keep_best=True`
recorded the weights at the epoch with the lowest validation loss, and
`core.restore_best` puts them back.

Early stopping is regularisation because a partly trained network is a *simpler*
function than a fully trained one. Training begins from small random weights and
grows them; stopping early is a way of keeping them small, which is what weight
decay does by a different route.

One practical warning. "Stop when the validation loss goes up" is a bad rule on a
noisy validation curve — it stops on the first upward flicker, often within a few
dozen epochs. The honest version is what `core.train` does: train to the end,
remember the best epoch, and go back to it.

### Your turn

Restore the best-validation weights of the **unregularised** run and measure it
again.

In [ ]:
# TODO 4 --- early stopping, after the fact ---------------------------------------------------
# Two `...` to replace, one per line:
#   model_early  ->  core.restore_best(core.MLP(hidden=(64, 64), activation="tanh"), history_big)
#   terr_early   ->  truth_error(model_early)
model_early = ...                                 # <- core.restore_best(core.MLP(hidden=(64, 64), activation="tanh"), history_big)
train_early = core.evaluate(model_early, x_train, y_train)
val_early   = core.evaluate(model_early, x_val, y_val)
tv_early    = total_variation(core.predict(model_early, xs))
terr_early  = ...                                 # <- truth_error(model_early)
# ------------------------------------------------------------------------------

In [ ]:
print(f"stopped at epoch      : {history_big['best_epoch']} of 6000")
print(f"training MSE          : {train_early:.6f}   (was {train_big:.8f})")
print(f"validation MSE        : {val_early:.5f}   (was {val_big:.5f})")
print(f"total variation       : {tv_early:.2f}   (was {tv_big:.2f}, truth {tv_truth:.2f})")
print(f"error against truth   : {terr_early:.5f}   (was {terr_big:.5f})")

**What you should see.** A training error that is *worse* by four orders of
magnitude — around 0.004 against $10^{-7}$ — a validation error that is better,
a total variation of about 0.83 against the overfit's 1.6, and an error against
the truth of roughly **0.0006**, which is eight or nine times smaller than the
overfitted model's.

Stopping at epoch two hundred of six thousand made the model nearly ten times
better at the job it was actually for, and made every number visible during
training worse.

Every single quantity you can measure without a held-out set says this model is
worse than the one from section 1. That is the whole lesson of L4.2 in one
comparison.

---

## 6 · How much was recovered?

Score the fixes on the **error against the truth**, not on the validation error.
Two reasons: its floor is zero, so a ratio between two models means something;
and it does not contain the forty points of measurement noise that make the
validation column hard to read.

The recovered fraction of a fix is then

$$ \text{recovery} \;=\; 1 \;-\;
   \frac{E_{\text{fix}}}{E_{\text{overfit}}},
   \qquad E = \overline{\bigl(f(x) - \text{truth}(x)\bigr)^2} $$

reported as a percentage. Be clear in your report that this number is available
only because the data is synthetic; the honest version of this table in a real
project uses the validation column and is much harder to interpret.

### Your turn

Compute the recovery for weight decay at your best $\lambda$ and for early
stopping.

In [ ]:
# TODO 5 --- the recovered fraction ----------------------------------------------------------
# Two `...` to replace, one per line:  recovery = 1 - truth_error(fixed) / truth_error(model_big)
#   rec_wd     ->  1.0 - truth_error(wd_models[best_lambda]) / terr_big
#   rec_early  ->  1.0 - terr_early / terr_big
rec_wd    = ...                                   # <- 1.0 - truth_error(wd_models[best_lambda]) / terr_big
rec_early = ...                                   # <- 1.0 - terr_early / terr_big
assert not any(v is ... for v in (rec_wd, rec_early)), "TODO 5: replace the two ..."
# ------------------------------------------------------------------------------

In [ ]:
rows = [
    ["none (section 1)", f"{val_big:.5f}", f"{terr_big:.5f}", f"{tv_big:.2f}", "0 %"],
    [f"weight decay {best_lambda:g}",
     f"{core.evaluate(wd_models[best_lambda], x_val, y_val):.5f}",
     f"{truth_error(wd_models[best_lambda]):.5f}",
     f"{total_variation(core.predict(wd_models[best_lambda], xs)):.2f}",
     f"{100 * rec_wd:.0f} %"],
    ["early stopping", f"{val_early:.5f}", f"{terr_early:.5f}", f"{tv_early:.2f}",
     f"{100 * rec_early:.0f} %"],
]
print(core.error_table(rows, ["regularisation", "validation MSE",
                              "error vs truth", "total variation",
                              "recovered"]))
print(f"\nnoise floor on the validation column: {NOISE_FLOOR:.5f}")
print(f"truth total variation: {tv_truth:.2f}")

**What you should see.** Both fixes recovering most of the error — typically
around ninety per cent for early stopping and rather more than that for weight
decay at its best value. Neither reaches 100 %, and neither should: eleven noisy
points do not contain enough information to recover the truth exactly, and no
amount of regularisation invents data.

Compare the two error columns as you read the table. The validation column moves
from about 0.0067 to about 0.0040 — a change of less than a factor of two, which
would be easy to dismiss. The truth column moves by a factor of ten or twenty.
Most of the validation column is noise in the held-out measurements, and it is
masking the size of the improvement. In a real project that masking is what you
have to work with.

The last sentence is the honest limit of this whole notebook. Regularisation
stops a model from claiming more than the data supports. It does not add
support.

---

## 7 · The three fits, together

Finally, rebuild the lecture's build — too flexible, about right — on one pair of
axes.

In [ ]:
fig, ax = plt.subplots(figsize=(9.0, 5.2))
core.plot_fit(x_train, y_train,
              {"1 — no regularisation": core.predict(model_big, xs),
               f"2 — weight decay {best_lambda:g}": core.predict(wd_models[best_lambda], xs),
               "3 — early stopping": core.predict(model_early, xs)},
              x_curve=xs, truth=core.lecture_truth,
              x_val=x_val, y_val=y_val, ax=ax, ylim=(-0.2, 1.5),
              title="One dataset, one architecture, three amounts of restraint")
plt.show()

**What you should see.** One curve swinging wildly through every training point;
two curves that stay near the truth and miss the training points by roughly the
size of the noise. All three came from the **same architecture with the same 4353
parameters**, trained on the same eleven points with the same optimiser.

That is the sentence to take away. Capacity is not a property of the parameter
count alone. It is a property of the parameter count *and* what you did while
fitting it, and the second half is under your control at no cost.

---

## 8 · An honesty note about the held-out set

You chose $\lambda$ by looking at the validation error, and you chose the
stopping epoch by looking at the validation error. The validation set has
therefore been used to make two decisions, and the number it reports for your
chosen model is now optimistic — it is, in a small way, a training set.

This is why the standard split has **three** parts, not two: train, validation,
test. The test set is touched once, at the end, after every decision has been
made. L4.2 slide 14 says so, and this notebook is a small demonstration of why.

With eleven training points and forty validation points, the optimism here is not
small. Say so in your report rather than quoting the validation number as if it
were an unbiased estimate of performance.

---

## 9 · Save the results

In [ ]:
os.makedirs(core.OUTPUT_DIR, exist_ok=True)
path = os.path.join(core.OUTPUT_DIR, "regularisation.npz")
np.savez(path,
         lambdas=np.array(LAMBDAS),
         val=np.array([core.evaluate(wd_models[l], x_val, y_val) for l in LAMBDAS]),
         train=np.array([core.evaluate(wd_models[l], x_train, y_train) for l in LAMBDAS]),
         tv=np.array([total_variation(core.predict(wd_models[l], xs)) for l in LAMBDAS]),
         truth_err=np.array([truth_error(wd_models[l]) for l in LAMBDAS]),
         best_lambda=best_lambda,
         val_overfit=val_big, val_early=val_early,
         recovery=np.array([rec_wd, rec_early]),
         noise_floor=NOISE_FLOOR)
print("saved:", path)

## 10 · Before you move on

1. The overfitted model had a training MSE of order $10^{-7}$ against a noise
   variance of $0.0049$. What, precisely, is the model representing in that
   last factor of ten thousand?
2. Early stopping and weight decay recovered similar amounts of the gap by
   different mechanisms. Describe each mechanism in one sentence, without using
   the word "complexity".
3. You have four hundred parameters per data point and the model still produced
   a usable fit once regularised. Reconcile that with the classical rule of
   thumb that you need more data points than parameters. (L4.2 slide 16 is the
   full answer; one sentence is enough here.)
4. Suppose you had no held-out set at all — eleven points and nothing else. Name
   two things you could still do to avoid the model in section 1, and say what
   each would cost you.

*Write your answers here.*

1.
2.
3.
4.

---

Continue with **`Ex04_06_report.ipynb`**.